# 05 - Fast evaluation with vLLM (base + every adapter found)

## 1. Installations

In [1]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
_vllm, _triton = ('vllm==0.11.2', 'triton') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
try:
    import importlib.metadata as _md; _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
except Exception:
    _torch_v = ()
_torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
!uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq math-verify
print("is_t4:", is_t4, "| vllm:", _vllm)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 78.2 MB/s eta 0:00:00
is_t4: True | vllm: vllm==0.11.2


## 2. Environment check

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_TF"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import importlib.metadata as md
for pkg in ["torch", "vllm", "transformers", "trl", "peft", "unsloth", "datasets", "math-verify"]:
    try: print(f"{pkg:13s} {md.version(pkg)}")
    except md.PackageNotFoundError: print(f"{pkg:13s} NOT INSTALLED")

import torch
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

torch         2.9.0
vllm          0.11.2
transformers  4.56.2
trl           0.22.2
peft          0.21.1
unsloth       2026.9.12
datasets      4.3.0
math-verify   0.9.0
GPU: Tesla T4 | 15.6 GB


## 3. Shared extractor + eval helpers

In [3]:
%%writefile rf_extract.py
import re

BOXED_SUFFIX = "\nPlease reason step by step, and put your final answer within \\boxed{}."


def user_prompt(question, style="boxed"):
    if style == "boxed":
        return question + BOXED_SUFFIX
    if style == "plain":
        return question
    raise ValueError(style)


def _boxed_spans(text):
    key = r"\boxed{"
    spans, i = [], 0
    while True:
        j = text.find(key, i)
        if j == -1:
            break
        k = j + len(key)
        start, depth = k, 1
        while k < len(text) and depth > 0:
            if text[k] == "{":
                depth += 1
            elif text[k] == "}":
                depth -= 1
            k += 1
        spans.append(text[start:k - 1])
        i = k
    return spans


def _unwrap_text(s):
    return re.sub(r"\\(?:text|textbf|mbox|mathrm)\{([^{}]*)\}", r"\1", s)


def norm(x):
    if x is None:
        return None
    s = str(x).strip()
    s = _unwrap_text(s)
    s = s.replace("\\!", "").replace("\\,", "").replace("\\;", "").replace(" ", "")
    s = s.replace("\\$", "").replace("$", "")
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = s.replace("\\%", "").replace("%", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("^\\circ", "").replace("^{\\circ}", "")
    s = s.rstrip(".").strip()
    if s.startswith("{") and s.endswith("}"):
        s = s[1:-1]
    return s


def extract_boxed(text):
    boxes = _boxed_spans(text)
    return norm(boxes[-1]) if boxes else None


def extract_answer(text):
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed
    nums = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return norm(nums[-1]) if nums else None


def _to_float(s):
    try:
        return float(s)
    except (TypeError, ValueError):
        return None


def _math_verify(pred, gold):
    try:
        from math_verify import parse, verify
    except ImportError:
        return False
    try:
        return bool(verify(parse("\\boxed{" + gold + "}"), parse("\\boxed{" + pred + "}")))
    except Exception:
        return False


def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    p, g = norm(pred), norm(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    gf = _to_float(g)
    if gf is not None:
        m = re.fullmatch(r"(-?\d+(?:\.\d+)?)[a-zA-Z]*", p)
        pf = _to_float(m.group(1)) if m else None
        return pf is not None and abs(pf - gf) < 1e-6
    return _math_verify(p, g)


def gsm8k_gold(answer_field):
    return norm(answer_field.split("####")[-1])


def has_boxed(text):
    return bool(_boxed_spans(text))

Writing rf_extract.py


In [4]:
%%writefile rf_eval.py
import os
import pickle

from rf_extract import answers_match, extract_answer, gsm8k_gold, has_boxed, user_prompt


def load_eval_set(name, n=None):
    from datasets import load_dataset
    if name == "gsm8k":
        ds = load_dataset("openai/gsm8k", "main", split="test")
        items = [{"question": r["question"], "gold": gsm8k_gold(r["answer"])} for r in ds]
    elif name == "math500":
        ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
        items = [{"question": r["problem"], "gold": r["answer"]} for r in ds]
    else:
        raise ValueError(name)
    return items[:n] if n else items


def chat_prompts(tokenizer, items, style):
    return [tokenizer.apply_chat_template([{"role": "user", "content": user_prompt(it["question"], style)}],
                                          tokenize=False, add_generation_prompt=True)
            for it in items]


def rows_from_outputs(items, outputs):
    rows = []
    for it, out in zip(items, outputs):
        c = out.outputs[0]
        rows.append({"question": it["question"], "gold": it["gold"], "resp": c.text,
                     "n_new_tokens": len(c.token_ids),
                     "truncated": int(c.finish_reason == "length"),
                     "has_box": has_boxed(c.text)})
    return rows


def quick_acc(rows):
    return sum(answers_match(extract_answer(r["resp"]), r["gold"]) for r in rows) / max(len(rows), 1)


def load_result(path, config):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {"config": config}


def save_result(path, result):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(result, f)
    os.replace(tmp, path)

Writing rf_eval.py


## 4. Config + adapter discovery

In [5]:
import glob, os, shutil

MODEL_ID  = "Qwen/Qwen2.5-1.5B-Instruct"
DATASETS  = ["gsm8k", "math500"]
STYLES    = ["boxed", "plain"]
MAX_NEW   = 4096
N_LIMIT   = None
OUT_DIR   = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)

for p in glob.glob("/kaggle/input/**/results/*.pkl", recursive=True):
    dst = os.path.join(OUT_DIR, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst); print("resuming from", p)

def short_name(path):
    s = path.lower()
    if "grpo" in s:
        return "grpo_b" if ("grpo_b" in s or "grpo-b" in s) else "grpo_a"
    if "sft" in s:
        return "sft"
    return os.path.basename(path.rstrip("/"))

ADAPTERS = {}
for cfg in sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)):
    d = os.path.dirname(cfg)
    if "checkpoint-" in d:
        continue
    ADAPTERS.setdefault(short_name(d), d)
print("adapters:", ADAPTERS if ADAPTERS else "NONE (base only) — did you Add Data?")

adapters: {'sft': '/kaggle/input/datasets/anjalii2511/sft-adapter'}


## 5. Load vLLM

In [6]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=MODEL_ID,
    dtype="half",
    max_model_len=MAX_NEW + 1024,
    gpu_memory_utilization=0.85,
    enable_lora=bool(ADAPTERS),
    max_lora_rank=64,
    max_loras=1,
    seed=0,
)
tokenizer = llm.get_tokenizer()

GREEDY = SamplingParams(temperature=0.0, max_tokens=MAX_NEW, repetition_penalty=1.0)
SAMPLED = SamplingParams(temperature=0.6, top_p=0.95, max_tokens=MAX_NEW, seed=0)
SAMPLED_FOR = ["base", "sft"]

RUNS = {"base": (None, GREEDY, STYLES)}
for i, (name, path) in enumerate(ADAPTERS.items(), start=1):
    RUNS[name] = (LoRARequest(name, i, path), GREEDY, STYLES)
for name in SAMPLED_FOR:
    if name in RUNS:
        RUNS[f"{name}_t06"] = (RUNS[name][0], SAMPLED, ["boxed"])
print("runs:", {k: v[2] for k, v in RUNS.items()})

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu128).
/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


INFO 10-01 06:03:21 [utils.py:253] non-default args: {'dtype': 'half', 'seed': 0, 'max_model_len': 5120, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 64, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

INFO 10-01 06:03:40 [model.py:631] Resolved architecture: Qwen2ForCausalLM
WARNING 10-01 06:03:40 [model.py:1971] Casting torch.bfloat16 to torch.float16.
INFO 10-01 06:03:40 [model.py:1745] Using max model len 5120
INFO 10-01 06:03:41 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

(EngineCore_DP0 pid=217) INFO 10-01 06:05:25 [backends.py:282] Compiling a graph for dynamic shape takes 20.30 s
(EngineCore_DP0 pid=217) INFO 10-01 06:05:27 [monitor.py:34] torch.compile takes 31.29 s in total
(EngineCore_DP0 pid=217) INFO 10-01 06:05:30 [gpu_worker.py:359] Available KV cache memory: 7.88 GiB
(EngineCore_DP0 pid=217) INFO 10-01 06:05:30 [kv_cache_utils.py:1229] GPU KV cache size: 294,960 tokens
(EngineCore_DP0 pid=217) INFO 10-01 06:05:30 [kv_cache_utils.py:1234] Maximum concurrency for 5,120 tokens per request: 57.61x


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/102 [00:00<?, ?it/s]

(EngineCore_DP0 pid=217) WARNING 10-01 06:05:31 [utils.py:250] Using default LoRA kernel configs


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:49<00:00,  2.08it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 70/70 [00:15<00:00,  4.65it/s]


(EngineCore_DP0 pid=217) INFO 10-01 06:06:35 [gpu_model_runner.py:4244] Graph capturing finished in 65 secs, took 0.97 GiB
(EngineCore_DP0 pid=217) INFO 10-01 06:06:35 [core.py:250] init engine (profile, create kv cache, warmup model) took 104.00 seconds
INFO 10-01 06:06:37 [llm.py:352] Supported tasks: ['generate']
runs: {'base': ['boxed', 'plain'], 'sft': ['boxed', 'plain'], 'base_t06': ['boxed'], 'sft_t06': ['boxed']}


## 6. Generate

In [7]:
import time
from rf_eval import load_eval_set, chat_prompts, rows_from_outputs, quick_acc, load_result, save_result

summary = []
for ds_name in DATASETS:
    items = load_eval_set(ds_name, N_LIMIT)
    for style in STYLES:
        path = f"{OUT_DIR}/{ds_name}_{style}.pkl"
        result = load_result(path, {"dataset": ds_name, "style": style, "n": len(items),
                                    "max_new": MAX_NEW, "model_id": MODEL_ID})
        prompts = chat_prompts(tokenizer, items, style)
        for name, (lora, sp, styles) in RUNS.items():
            if style not in styles:
                continue
            if len(result.get(name, [])) == len(items):
                print(f"[skip] {ds_name}/{style}/{name} already done")
            else:
                t0 = time.time()
                outs = llm.generate(prompts, sp, lora_request=lora)
                result[name] = rows_from_outputs(items, outs)
                result["config"].setdefault("decoding", {})[name] = f"temperature={sp.temperature}, top_p={sp.top_p}"
                save_result(path, result)
                print(f"[done] {ds_name}/{style}/{name}: {(time.time()-t0)/60:.1f} min")
            rows = result[name]
            summary.append((ds_name, style, name, quick_acc(rows),
                            sum(r["truncated"] for r in rows) / len(rows),
                            sum(r["has_box"] for r in rows) / len(rows)))

print(f"\n{'dataset':8s} {'style':6s} {'model':8s} {'acc':>6s} {'trunc':>6s} {'boxed':>6s}")
for d, s, m, a, t, b in summary:
    print(f"{d:8s} {s:6s} {m:8s} {a*100:5.1f}% {t*100:5.1f}% {b*100:5.1f}%")

[2026-10-01 06:06:37] INFO config.py:114: Disabling Tensorflow because USE_TORCH is set
[2026-10-01 06:06:37] INFO config.py:125: JAX version 0.7.2 available.


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

Processed prompts:   0%|          | 0/500 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[done] math500/plain/base: 6.4 min


Adding requests:   0%|          | 0/500 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/500 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[done] math500/plain/sft: 69.0 min

dataset  style  model       acc  trunc  boxed
gsm8k    boxed  base      75.7%   0.3%  99.2%
gsm8k    boxed  sft       61.3%  24.9%  75.7%
gsm8k    boxed  base_t06  74.3%   0.0%  99.2%
gsm8k    boxed  sft_t06   58.4%  16.7%  84.7%
gsm8k    plain  base      71.8%   0.5%  71.2%
gsm8k    plain  sft       56.8%  32.0%  68.2%
math500  boxed  base      56.4%   5.0%  95.0%
math500  boxed  sft       31.6%  52.0%  50.4%
math500  boxed  base_t06  55.0%   1.8%  98.0%
math500  boxed  sft_t06   30.6%  33.0%  74.2%
math500  plain  base      53.0%   4.8%  91.8%
math500  plain  sft       32.4%  54.6%  46.4%


## 7. Package for download

In [8]:
shutil.make_archive("/kaggle/working/results", "zip", "/kaggle/working", "results")
print("download /kaggle/working/results.zip from the Output panel")

download /kaggle/working/results.zip from the Output panel
